In [17]:
!pip install -q pypdf
!pip -q install -U \
langchain \
langchain-community \
langchain-text-splitters \
langchain-chroma \
langchain-google-genai \
sentence-transformers \
unstructured \
chromadb

In [ ]:
import os

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

In [19]:
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader

loader = BSHTMLLoader(
    "How to use the various modes of the washing machine _ Samsung LEVANT.html",
    open_encoding="utf-8",
    bs_kwargs={"features": "html.parser"}
)
documents = loader.load()
print(documents[0].page_content[:500])

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

In [22]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

In [24]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [25]:
prompt = ChatPromptTemplate.from_template(
"""
You are an expert Samsung Washing Machine assistant.

Answer ONLY using the provided context.

If the answer is not found in the context,
say:

"I couldn't find that information in the manual."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [27]:
def ask(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    messages = prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(messages)

    return response.content

In [ ]:
print(
    ask("What does Eco Wash mode do?")
)